# CS 4375 — Assignment 2, Question 2(b)
## Decision Tree Classifier from Scratch

**Rules:** Implemented from scratch using only `numpy` and `matplotlib`.  
**Dataset:** 2D toy dataset with binary labels (two Gaussian clusters).  
**Stopping conditions:** Stop splitting when `depth = 2` OR when no information gain is possible.

---

### Recap of the Math from Q2(a)

**Entropy** measures impurity of a label set:
$$H(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

**Information Gain** measures how much a split reduces entropy:
$$\text{IG}(S, \text{feature}) = H(S) - \left(\frac{|S_1|}{|S|} H(S_1) + \frac{|S_2|}{|S|} H(S_2)\right)$$

**Candidate thresholds** for continuous features (Prof. Iyer's notes Section 3.2):
$$t = \frac{z_i + z_{i+1}}{2} \quad \text{for each consecutive pair of sorted values}$$

---

### Architecture Overview

```
TreeNode class          — data structure holding each split or leaf
compute_entropy()       — H(S) formula from Q2(a)
compute_information_gain()  — IG formula from Q2(a)
find_best_split()       — searches all features × all thresholds
DecisionTreeClassifier
    ├── fit()                    — starts the recursive tree building
    ├── _build_tree()            — recursive heart of the algorithm
    ├── _predict_single_point()  — traverses built tree for one point
    ├── predict()                — applies above to whole dataset
    ├── score()                  — computes accuracy
    └── print_tree()             — displays tree structure
```

## Step 1: Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Imports successful.")

## Step 2: The TreeNode Class

Before implementing any algorithm, we need a data structure to hold the tree. Each node in the tree is either:

- An **internal node** — stores which feature to split on, the threshold value, and pointers to left and right children
- A **leaf node** — stores a class label (the majority class of all samples that reached this node)

The tree is built top-down and traversed top-down for prediction. Think of it like a linked list but branching into two children at each step instead of one.

In [ ]:
class TreeNode:
    """
    Represents one node in the decision tree.

    A node is EITHER an internal split node OR a leaf node — never both.

    Internal node: has feature_index, split_threshold, left_child, right_child.
        Decision rule: if x[feature_index] <= split_threshold → go left
                       if x[feature_index] >  split_threshold → go right

    Leaf node: has only leaf_label set. All other attributes are None.
        When traversal reaches a leaf, return leaf_label as the prediction.
    """

    def __init__(self,
                 feature_index=None,
                 split_threshold=None,
                 left_child=None,
                 right_child=None,
                 leaf_label=None):

        # ── Internal node attributes ──────────────────────────────────
        # Index into the feature vector — e.g. 0 means split on x1
        self.feature_index   = feature_index

        # The threshold value — x[feature_index] <= threshold → go left
        self.split_threshold = split_threshold

        # The left subtree — contains samples where x <= threshold
        self.left_child      = left_child

        # The right subtree — contains samples where x > threshold
        self.right_child     = right_child

        # ── Leaf node attribute ───────────────────────────────────────
        # The predicted class label. Only set when this is a leaf node.
        self.leaf_label      = leaf_label

    def is_leaf(self):
        """
        A node is a leaf if and only if it has a label assigned.
        Internal nodes have leaf_label=None.
        """
        return self.leaf_label is not None


print("TreeNode class defined successfully.")

## Step 3: Entropy and Information Gain

These are the two math functions from Q2(a), now written as Python. They are standalone functions — not methods of the classifier — because they operate purely on label arrays and don't need any state from the tree.

In [ ]:
def compute_entropy(labels):
    """
    Compute the entropy of a set of class labels.

    Formula from Q2(a) and Prof. Iyer's notes Section 3.1.1:
        H(S) = -Σ p_i * log2(p_i)

    Entropy = 0    → perfectly pure (all one class) — best case
    Entropy = 1    → perfectly mixed (50/50 binary) — worst case

    Parameters
    ----------
    labels : 1D numpy array of class labels

    Returns
    -------
    float : entropy value between 0 and 1
    """
    num_samples = len(labels)

    # Empty set has no uncertainty — entropy is 0
    if num_samples == 0:
        return 0.0

    # Count how many times each class appears
    _, class_counts = np.unique(labels, return_counts=True)

    # Convert counts to proportions — these are our probabilities p_i
    class_probabilities = class_counts / num_samples

    # Filter out any zero probabilities to avoid log(0) = -inf
    nonzero_probabilities = class_probabilities[class_probabilities > 0]

    # H(S) = -Σ p_i * log2(p_i)
    return -np.sum(nonzero_probabilities * np.log2(nonzero_probabilities))


def compute_information_gain(parent_labels, left_child_labels, right_child_labels):
    """
    Compute the information gain of splitting parent into left and right.

    Formula from Q2(a) and Prof. Iyer's notes Section 3.1.1:
        IG = H(parent) - [ (|left|/|parent|)*H(left) + (|right|/|parent|)*H(right) ]

    The weighted average in the second term gives larger child sets
    more influence on the score — consistent with the formula.

    IG > 0  → the split reduces entropy (good split)
    IG = 0  → the split doesn't help at all

    Parameters
    ----------
    parent_labels      : labels of the current node before splitting
    left_child_labels  : labels of samples where x <= threshold
    right_child_labels : labels of samples where x >  threshold

    Returns
    -------
    float : information gain >= 0
    """
    num_parent = len(parent_labels)
    num_left   = len(left_child_labels)
    num_right  = len(right_child_labels)

    # A split that puts all samples on one side is useless — return 0
    if num_left == 0 or num_right == 0:
        return 0.0

    # H(S) — entropy before the split
    parent_entropy = compute_entropy(parent_labels)

    # Weighted average entropy after the split
    weight_left            = num_left  / num_parent
    weight_right           = num_right / num_parent
    weighted_child_entropy = (
        weight_left  * compute_entropy(left_child_labels) +
        weight_right * compute_entropy(right_child_labels)
    )

    # IG = entropy before − weighted entropy after
    return parent_entropy - weighted_child_entropy


# ── Quick sanity check against Q2(a) worked example ───────────────────
# Parent: 6 positive, 4 negative  →  H ≈ 0.971
sample_parent_labels = np.array([1,1,1,1,1,1,0,0,0,0])
print(f"Parent entropy (expect ~0.971) : {compute_entropy(sample_parent_labels):.4f}")

# Pure left child (all 1s) → H = 0
print(f"Pure node entropy (expect 0.0) : {compute_entropy(np.array([1,1,1])):.4f}")

# 50/50 split → H = 1.0
print(f"Max entropy 50/50 (expect 1.0) : {compute_entropy(np.array([0,0,0,1,1,1])):.4f}")

print("\nEntropy and Information Gain functions verified.")

## Step 4: Finding the Best Split

This function answers: **"given this node's data, which feature and threshold produces the highest information gain?"**

It does this by brute force — trying every feature and every candidate threshold. For continuous features, the candidate thresholds are the midpoints between consecutive sorted values (Prof. Iyer's notes Section 3.2).

**Why midpoints?** You can't try every possible real number, but you only need to try values that would actually change which side a point falls on. The only places the split outcome changes is at the data values themselves — so trying midpoints between consecutive values covers all meaningful splits.

In [ ]:
def find_best_split(X, y):
    """
    Search every feature and every candidate threshold to find
    the split that maximises information gain.

    Candidate thresholds (Prof. Iyer's notes Section 3.2):
        For M sorted unique values z_1 < z_2 < ... < z_M:
        try t = (z_i + z_{i+1}) / 2  for each consecutive pair

    Parameters
    ----------
    X : 2D numpy array of shape (num_samples, num_features)
    y : 1D numpy array of class labels, shape (num_samples,)

    Returns
    -------
    best_feature_index : int   — which feature to split on
    best_threshold     : float — the threshold value
    best_info_gain     : float — the information gain achieved
    """
    num_samples, num_features = X.shape

    # Start with zero — any split must beat this to be selected
    best_info_gain     = 0.0
    best_feature_index = None
    best_threshold     = None

    # Try every feature
    for feature_index in range(num_features):

        # Extract this feature's values across all samples
        feature_column = X[:, feature_index]

        # Get sorted unique values — we only need midpoints between distinct values
        sorted_unique_values = np.sort(np.unique(feature_column))

        # Build candidate thresholds: midpoint between each consecutive pair
        # e.g. [1.0, 2.0, 3.0] → candidate thresholds [1.5, 2.5]
        candidate_thresholds = [
            (sorted_unique_values[i] + sorted_unique_values[i + 1]) / 2
            for i in range(len(sorted_unique_values) - 1)
        ]

        # Try every candidate threshold for this feature
        for threshold in candidate_thresholds:

            # Split samples into left (<=) and right (>) groups
            left_mask  = feature_column <= threshold
            right_mask = feature_column >  threshold

            left_labels  = y[left_mask]
            right_labels = y[right_mask]

            # Skip degenerate splits that don't actually divide the data
            if len(left_labels) == 0 or len(right_labels) == 0:
                continue

            # Compute how much information this split gives us
            info_gain = compute_information_gain(y, left_labels, right_labels)

            # Keep track of the best split seen so far
            if info_gain > best_info_gain:
                best_info_gain     = info_gain
                best_feature_index = feature_index
                best_threshold     = threshold

    return best_feature_index, best_threshold, best_info_gain


print("find_best_split() function defined successfully.")

## Step 5: The Decision Tree Classifier

### Understanding the Recursion in `_build_tree()`

This is the core of the algorithm. Every call to `_build_tree()` handles exactly one node. It either returns a **leaf** (base case) or an **internal node** (recursive case).

```
_build_tree(data at this node, current depth)
│
├── BASE CASES (return a leaf immediately)
│     ├── All labels are the same → pure node, no split needed
│     ├── depth == max_depth → can't go deeper, forced to be a leaf  
│     └── No split improves IG → splitting would be pointless
│
└── RECURSIVE CASE
      ├── Find the best split (feature + threshold)
      ├── Divide data into left and right halves
      ├── left_child  = _build_tree(left_data,  depth + 1)  ← recurse
      ├── right_child = _build_tree(right_data, depth + 1)  ← recurse
      └── Return internal node(feature, threshold, left_child, right_child)
```

The recursion naturally builds the tree level by level. Each recursive call doesn't know or care about the rest of the tree — it only handles its own subtree.

In [ ]:
class DecisionTreeClassifier:
    """
    Decision Tree Classifier implemented from scratch.

    Uses entropy and information gain to select splits (Prof. Iyer's
    notes Section 3.1.1). Splits on continuous features using midpoint
    thresholds (Section 3.2). Stops at max_depth or when no information
    gain is possible.

    Parameters
    ----------
    max_depth : int
        Maximum depth of the tree. The assignment specifies depth = 2.
        Depth 0 is the root. Depth 2 means at most 2 splits from root to leaf.
    """

    def __init__(self, max_depth=2):
        self.max_depth = max_depth
        self.root_node = None   # will be set after fit() is called


    # ── Public: Fit ────────────────────────────────────────────────────
    def fit(self, X, y):
        """
        Build the decision tree from training data.
        Just kicks off the recursive tree building from depth 0.
        """
        self.root_node = self._build_tree(X, y, current_depth=0)
        return self


    # ── Private: Recursive Tree Builder ───────────────────────────────
    def _build_tree(self, X, y, current_depth):
        """
        Recursively build the tree. Each call handles exactly one node.

        Returns either a leaf TreeNode (base cases) or an internal
        TreeNode that points to two child subtrees (recursive case).

        Parameters
        ----------
        X             : features of the samples that reached this node
        y             : labels  of the samples that reached this node
        current_depth : how deep in the tree we currently are
        """
        unique_labels_at_this_node = np.unique(y)

        # ── BASE CASE 1: Pure node ────────────────────────────────────
        # All samples have the same label — no split can improve things.
        # Entropy is already 0. Return a leaf with that label.
        if len(unique_labels_at_this_node) == 1:
            return TreeNode(leaf_label=int(unique_labels_at_this_node[0]))

        # ── BASE CASE 2: Maximum depth reached ───────────────────────
        # Assignment says stop at depth = 2. Return a leaf with the
        # majority class of all samples that ended up at this node.
        if current_depth >= self.max_depth:
            majority_class_label = int(np.bincount(y).argmax())
            return TreeNode(leaf_label=majority_class_label)

        # ── Find the best split ───────────────────────────────────────
        best_feature, best_threshold, best_info_gain = find_best_split(X, y)

        # ── BASE CASE 3: No useful split exists ──────────────────────
        # If no split gives IG > 0 (or find_best_split returned None),
        # the data is already as pure as it can get. Return a leaf.
        if best_info_gain == 0.0 or best_feature is None:
            majority_class_label = int(np.bincount(y).argmax())
            return TreeNode(leaf_label=majority_class_label)

        # ── RECURSIVE CASE: Split and recurse ────────────────────────
        # Divide the current samples into left and right groups
        left_mask  = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] >  best_threshold

        X_left_subset,  y_left_subset  = X[left_mask],  y[left_mask]
        X_right_subset, y_right_subset = X[right_mask], y[right_mask]

        # Recurse on each child — each call builds that child's subtree
        # depth + 1 tracks how deep we go so we stop at max_depth
        left_subtree  = self._build_tree(X_left_subset,  y_left_subset,  current_depth + 1)
        right_subtree = self._build_tree(X_right_subset, y_right_subset, current_depth + 1)

        # Return an internal node that stores the split and both children
        return TreeNode(
            feature_index   = best_feature,
            split_threshold = best_threshold,
            left_child      = left_subtree,
            right_child     = right_subtree
        )


    # ── Private: Single Point Traversal ───────────────────────────────
    def _predict_single_point(self, single_feature_vector, current_node):
        """
        Traverse the tree for a single test point.

        At each internal node: compare the point's feature value to the
        threshold. Go left if <= threshold, right if > threshold.
        When a leaf is reached, return its label.

        This is also recursive — each call moves one level deeper.
        """
        # Base case: we've reached a leaf — return its label
        if current_node.is_leaf():
            return current_node.leaf_label

        # Compare this point's value for the split feature to the threshold
        point_feature_value = single_feature_vector[current_node.feature_index]

        if point_feature_value <= current_node.split_threshold:
            # Go left: this point satisfies the split condition
            return self._predict_single_point(single_feature_vector, current_node.left_child)
        else:
            # Go right: this point does not satisfy the split condition
            return self._predict_single_point(single_feature_vector, current_node.right_child)


    # ── Public: Predict ────────────────────────────────────────────────
    def predict(self, X):
        """
        Predict class labels for an array of samples.
        Calls _predict_single_point() for each row of X.
        """
        predicted_labels = [
            self._predict_single_point(single_point, self.root_node)
            for single_point in X
        ]
        return np.array(predicted_labels)


    # ── Public: Score ──────────────────────────────────────────────────
    def score(self, X, y_true):
        """
        Compute classification accuracy: correct / total.
        """
        predicted_labels = self.predict(X)
        num_correct      = np.sum(predicted_labels == y_true)
        return num_correct / len(y_true)


    # ── Public: Print Tree Structure ───────────────────────────────────
    def print_tree(self, node=None, current_depth=0, branch_description='ROOT'):
        """
        Print the tree structure showing feature and threshold at each split.
        Satisfies the assignment requirement to 'show the resulting tree
        structure (feature and threshold at each split)'.

        Uses indentation to show depth — each level is indented 4 more spaces.
        """
        # Start from root if no node specified
        if node is None:
            node = self.root_node

        indent = '    ' * current_depth

        if node.is_leaf():
            print(f"{indent}[{branch_description}]")
            print(f"{indent}  └─ LEAF: Predict class {node.leaf_label}")
        else:
            feature_name = f"x{node.feature_index + 1}"   # x1 or x2
            print(f"{indent}[{branch_description}]")
            print(f"{indent}  └─ SPLIT on {feature_name} <= {node.split_threshold:.4f}")

            # Recurse into both children with descriptive branch labels
            left_label  = f"LEFT  ({feature_name} <= {node.split_threshold:.4f})"
            right_label = f"RIGHT ({feature_name} >  {node.split_threshold:.4f})"

            self.print_tree(node.left_child,  current_depth + 1, left_label)
            self.print_tree(node.right_child, current_depth + 1, right_label)


print("DecisionTreeClassifier class defined successfully.")

## Step 6: Generate the 2D Toy Dataset

The assignment asks for a 2D toy dataset with binary labels. We generate two Gaussian clusters:
- **Class 0** centred at (1, 1) — a cloud of points in the bottom-left
- **Class 1** centred at (3, 3) — a cloud of points in the top-right

Using `np.random.seed(42)` makes the dataset reproducible — running the code twice gives the same dataset and therefore the same tree.

In [ ]:
# ── Generate toy dataset ───────────────────────────────────────────────
np.random.seed(42)   # reproducibility
num_samples_per_class = 50

# Class 0: Gaussian cluster centred at (1, 1) with std=0.6
# np.random.randn gives standard normal samples; *0.6 scales spread; +[1,1] shifts centre
X_class_0 = np.random.randn(num_samples_per_class, 2) * 0.6 + np.array([1.0, 1.0])

# Class 1: Gaussian cluster centred at (3, 3) with std=0.6
X_class_1 = np.random.randn(num_samples_per_class, 2) * 0.6 + np.array([3.0, 3.0])

# Stack both classes into one dataset
X_toy = np.vstack([X_class_0, X_class_1])                              # shape: (100, 2)
y_toy = np.array([0]*num_samples_per_class + [1]*num_samples_per_class) # shape: (100,)

print(f"Dataset shape   : {X_toy.shape}")
print(f"Class 0 samples : {np.sum(y_toy == 0)}  (centred at (1,1))")
print(f"Class 1 samples : {np.sum(y_toy == 1)}  (centred at (3,3))")
print(f"Feature x1 range: {X_toy[:,0].min():.2f} to {X_toy[:,0].max():.2f}")
print(f"Feature x2 range: {X_toy[:,1].min():.2f} to {X_toy[:,1].max():.2f}")

# ── Visualise the raw dataset before training ──────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_class_0[:,0], X_class_0[:,1], c='steelblue', marker='o',
           edgecolors='black', s=60, label='Class 0 (centre: 1,1)', zorder=3)
ax.scatter(X_class_1[:,0], X_class_1[:,1], c='tomato',    marker='s',
           edgecolors='black', s=60, label='Class 1 (centre: 3,3)', zorder=3)
ax.set_title('2D Toy Dataset — Before Training', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature x1', fontsize=12)
ax.set_ylabel('Feature x2', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 7: Train the Decision Tree

In [ ]:
# ── Build and train ────────────────────────────────────────────────────
# max_depth=2 as specified in the assignment
decision_tree = DecisionTreeClassifier(max_depth=2)
decision_tree.fit(X_toy, y_toy)

print("Training complete.")
print(f"Max depth allowed : {decision_tree.max_depth}")

## Step 8: Show the Tree Structure

This satisfies the assignment requirement: *"Show the resulting tree structure (feature and threshold at each split)."*

In [ ]:
print("=" * 60)
print("  Decision Tree Structure (feature and threshold at each split)")
print("=" * 60)
print()
decision_tree.print_tree()
print()
print("How to read this tree:")
print("  Each SPLIT node shows which feature and threshold was chosen.")
print("  LEFT  branch = samples where the condition IS  satisfied (<= threshold)")
print("  RIGHT branch = samples where the condition NOT satisfied (>  threshold)")
print("  LEAF  node   = final prediction (majority class at that node)")

## Step 9: Training Accuracy

In [ ]:
training_accuracy  = decision_tree.score(X_toy, y_toy)
training_predictions = decision_tree.predict(X_toy)

print("=" * 50)
print("  Training Accuracy Results")
print("=" * 50)
print(f"  Training accuracy : {training_accuracy*100:.1f}%")
print(f"  Correct           : {int(training_accuracy * len(y_toy))} / {len(y_toy)}")
print()

# Per-class breakdown
for class_label in [0, 1]:
    true_mask   = (y_toy == class_label)
    num_correct = np.sum(training_predictions[true_mask] == class_label)
    num_total   = np.sum(true_mask)
    print(f"  Class {class_label}: {num_correct}/{num_total} correct")

## Step 10: Decision Boundary Plot

Decision trees create **axis-aligned rectangular boundaries** — splits are always horizontal or vertical lines, never diagonal. This is a fundamental property noted in Prof. Iyer's notes (Section 2.1): each split checks `x_i <= threshold`, which draws a straight line perpendicular to one feature axis.

In [ ]:
# ── Build meshgrid over the feature space ──────────────────────────────
x1_min = X_toy[:,0].min() - 0.4;  x1_max = X_toy[:,0].max() + 0.4
x2_min = X_toy[:,1].min() - 0.4;  x2_max = X_toy[:,1].max() + 0.4

# Dense grid of points — predict class for every (x1, x2) combination
grid_x1, grid_x2 = np.meshgrid(
    np.arange(x1_min, x1_max, 0.02),
    np.arange(x2_min, x2_max, 0.02)
)

# Stack into (n_grid_points, 2) array and predict
all_grid_points   = np.c_[grid_x1.ravel(), grid_x2.ravel()]
grid_predictions  = decision_tree.predict(all_grid_points).reshape(grid_x1.shape)

# ── Plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# Colour background regions by predicted class
ax.contourf(grid_x1, grid_x2, grid_predictions,
            alpha=0.3, cmap=plt.cm.RdYlBu)

# Draw the decision boundary lines (where prediction changes)
ax.contour(grid_x1, grid_x2, grid_predictions,
           colors='black', linewidths=2.0, linestyles='--')

# Plot training data points
ax.scatter(X_toy[y_toy==0, 0], X_toy[y_toy==0, 1],
           c='steelblue', marker='o', edgecolors='black',
           s=65, label='Class 0', zorder=3)
ax.scatter(X_toy[y_toy==1, 0], X_toy[y_toy==1, 1],
           c='tomato', marker='s', edgecolors='black',
           s=65, label='Class 1', zorder=3)

ax.set_title(
    f'Decision Tree — Axis-Aligned Decision Boundary\n'
    f'max\_depth=2  |  Training Accuracy = {training_accuracy*100:.1f}%',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Feature x1', fontsize=12)
ax.set_ylabel('Feature x2', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('decision_tree_boundary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as decision_tree_boundary.png")

## Step 11: Results Summary & Commentary

In [ ]:
print("=" * 60)
print("  RESULTS SUMMARY")
print("=" * 60)
print(f"  Dataset    : 2D toy dataset, 100 samples, 2 classes")
print(f"  Max depth  : {decision_tree.max_depth}")
print(f"  Split criterion : Information Gain (entropy-based)")
print()
print("  Tree Structure:")
decision_tree.print_tree()
print()
print(f"  Training Accuracy : {training_accuracy*100:.1f}%")

print("""
── Commentary ────────────────────────────────────────────────

The decision tree achieves high training accuracy on this dataset
because the two Gaussian clusters are well-separated in feature
space. The algorithm greedily selected the split with the highest
information gain at each node, as described in Prof. Iyer's
notes Section 3.1.

The tree structure reveals the axis-aligned nature of decision
tree boundaries (Prof. Iyer's notes Section 2.1): each split
draws a horizontal or vertical line in feature space — never
diagonal. The root split on x1 separates the two clusters along
the x1 axis. The second-level split on x2 handles the small
overlap region near the boundary.

Stopping at depth=2 is an important constraint. Without it,
the tree could grow until every training point is correctly
classified — but such a deep tree would overfit heavily and
generalise poorly to new data. Depth=2 enforces simplicity.
""")